# 05 — Daily Signal Generation (Ensemble)

Production notebook for generating daily trading signals with ensemble predictions
(XGBoost + TFT). Loads trained models, fetches latest market data, computes features,
and outputs a CSV with signal direction, confidence, and Kelly-sized position.

**Output:** `signals.csv` with columns: pair, signal, confidence, position_size, timestamp

**Requires:** xgb_model.joblib + optimal_threshold.npy + ensemble_weight.npy from Notebook 03.

In [ ]:
!pip install yfinance --quiet

## Configuration

In [ ]:
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, "/kaggle/input/forex-ml-02-feature-engineering")

# Inline indicator functions (no pandas-ta dependency)
def _sma(s, p): return s.rolling(p).mean()
def _ema(s, p): r = s.ewm(span=p, adjust=False).mean(); r.iloc[:p-1] = np.nan; return r
def _rsi(s, p=14):
    d = s.diff(); g = d.where(d>0,0); l = (-d).where(d<0,0)
    ag = g.ewm(span=p, adjust=False).mean(); al = l.ewm(span=p, adjust=False).mean()
    rs = ag / al.replace(0, np.nan); return 100 - (100 / (1 + rs))
def _macd(s, f=12, sl=26, sg=9):
    m = _ema(s,f) - _ema(s,sl); return {"histogram": m - _ema(m,sg)}
def _bb(s, p=20, std=2.0):
    m = _sma(s,p); sd = s.rolling(p).std()
    return {"middle": m, "upper": m+sd*std, "lower": m-sd*std}
def _atr(h, l, c, p=14):
    tr = pd.concat([(h-l).abs(), (h-c.shift(1)).abs(), (l-c.shift(1)).abs()], axis=1).max(axis=1)
    return tr.rolling(p).mean()
def _vol_regime(atr_s, lo=0.3, hi=0.7):
    lt = atr_s.quantile(lo); ht = atr_s.quantile(hi)
    r = pd.Series(1.0, index=atr_s.index); r[atr_s<=lt] = 0.0; r[atr_s>=ht] = 2.0; r[atr_s.isna()] = np.nan
    return r

import yfinance as yf
import joblib
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

CURRENCY_PAIRS = [
    "EURUSD", "GBPUSD", "USDJPY", "USDCAD", "AUDUSD",
    "NZDUSD", "USDCHF", "EURGBP", "EURJPY", "EURCHF",
]
PAIR_ID_MAP = {p: i for i, p in enumerate(CURRENCY_PAIRS)}

ACCOUNT_VALUE = 10000.0
RISK_PER_TRADE = 0.02
KELLY_FRACTION = 0.25
LOOKBACK_DAYS = 365

EXCLUDE_COLS = {"open", "high", "low", "close", "volume", "adj_close", "pair", "pair_id"}

print("Configuration loaded")

## Helper functions

In [ ]:
def compute_indicators(sub):
    close, high, low = sub["close"], sub["high"], sub["low"]
    out = sub.copy()

    out["RSI"] = _rsi(close, 14)
    macd_res = _macd(close, 12, 26, 9)
    if macd_res is not None:
        out["MACD_Hist"] = macd_res["histogram"]
    bb = _bb(close, 20, 2.0)
    if bb is not None:
        denom = (bb["upper"] - bb["lower"]).clip(lower=1e-8)
        out["BB_PctB"] = (close - bb["lower"]) / denom
        out["BB_Width"] = (bb["upper"] - bb["lower"]) / bb["middle"].clip(lower=1e-8)
    out["ATR"] = _atr(high, low, close, 14)
    out["SMA_20"] = _sma(close, 20)
    out["SMA_50"] = _sma(close, 50)
    ema_fast, ema_slow = _ema(close, 12), _ema(close, 26)
    if ema_slow is not None and (ema_slow != 0).any():
        out["MA_Distance"] = (close - ema_slow) / ema_slow
    out["Body_Ratio"] = (close - sub["open"]) / (high - low).clip(lower=1e-8)
    out["Upper_Wick"] = (high - sub[["open", "close"]].max(axis=1)) / (high - low).clip(lower=1e-8)
    out["Lower_Wick"] = (sub[["open", "close"]].min(axis=1) - low) / (high - low).clip(lower=1e-8)
    out["Log_Returns"] = np.log(close / close.shift(1).clip(lower=1e-8))

    # Volatility regime
    out["Vol_Regime"] = _vol_regime(out["ATR"], 0.3, 0.7)
    # Rolling stats
    out["Ret_Skew_20"] = out["Log_Returns"].rolling(20).skew()
    out["Ret_Kurt_20"] = out["Log_Returns"].rolling(20).kurt()
    out["Ret_Mean_5"] = out["Log_Returns"].rolling(5).mean()
    out["Ret_Std_5"] = out["Log_Returns"].rolling(5).std()
    # Lags
    for lag in [1, 2, 3, 5]:
        out[f"RSI_Lag{lag}"] = out["RSI"].shift(lag)
        out[f"BB_PctB_Lag{lag}"] = out["BB_PctB"].shift(lag)
        out[f"ATR_Lag{lag}"] = out["ATR"].shift(lag)

    return out


def get_feature_cols(df):
    return [c for c in df.columns if c not in EXCLUDE_COLS]


def kelly_criterion(win_rate, avg_win, avg_loss):
    if avg_loss == 0: return 0.0
    r = avg_win / abs(avg_loss)
    return max(0.0, win_rate - (1 - win_rate) / r)

def fractional_kelly(win_rate, avg_win, avg_loss, fraction=KELLY_FRACTION):
    return kelly_criterion(win_rate, avg_win, avg_loss) * fraction

def position_size(account_value, confidence, win_rate, avg_win, avg_loss):
    kelly = fractional_kelly(win_rate, avg_win, avg_loss)
    return account_value * RISK_PER_TRADE * kelly * confidence


print("Helper functions defined")

In [ ]:
import numpy as np
from numpy._core import umath as _umath
patched = 0
for _name in ["_center", "_expandtabs", "_expandtabs_length", "_ljust",
              "_lstrip_chars", "_lstrip_whitespace", "_partition",
              "_partition_index", "_replace", "_rjust", "_rpartition",
              "_rpartition_index", "_rstrip_chars", "_rstrip_whitespace",
              "_slice", "_strip_chars", "_strip_whitespace", "_zfill"]:
    if not hasattr(_umath, _name):
        setattr(_umath, _name, object())
        patched += 1
print(f"Patched {patched} missing ufuncs" if patched else "All ufuncs present")

## Load model and threshold

In [ ]:
model = joblib.load("/kaggle/input/forex-ml-03-model-training/xgb_model.joblib")
threshold = float(np.load("/kaggle/input/forex-ml-03-model-training/optimal_threshold.npy"))
print(f"XGBoost model loaded. Threshold: {threshold:.2f}")

# Load ensemble weight (defaults to 1.0 i.e. XGBoost-only)
try:
    ensemble_weight = float(np.load("/kaggle/input/forex-ml-03-model-training/ensemble_weight.npy"))
    print(f"Ensemble weight (XGBoost): {ensemble_weight:.3f}")
except FileNotFoundError:
    ensemble_weight = 1.0
    print("No ensemble weight found — using XGBoost only")

## Fetch latest market data

In [ ]:
end_date = datetime.now()
start_date = end_date - timedelta(days=LOOKBACK_DAYS + 60)

frames = []
for pair in CURRENCY_PAIRS:
    try:
        df = yf.download(f"{pair}=X", start=start_date, end=end_date, interval="1d", progress=False)
        if df.empty:
            print(f"{pair}: empty DataFrame")
            continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        df["pair"] = pair
        df["pair_id"] = PAIR_ID_MAP[pair]
        frames.append(df)
        print(f"{pair}: {len(df)} rows")
    except Exception as e:
        print(f"{pair}: FAILED \u2014 {e}")

raw = pd.concat(frames).sort_index()
print(f"\nTotal: {len(raw)} rows across {len(CURRENCY_PAIRS)} pairs")

## Compute features + DXY correlation

In [ ]:
# Fetch DXY for correlation feature
try:
    dxy = yf.download("DX-Y.NYB", start=start_date, end=end_date, interval="1d", progress=False)
    if isinstance(dxy.columns, pd.MultiIndex):
        dxy.columns = dxy.columns.get_level_values(0)
    dxy = dxy["Close"].squeeze().rename("DXY")
    dxy.index = pd.to_datetime(dxy.index)
    print(f"DXY: {len(dxy)} rows")
except Exception as e:
    print(f"DXY fetch failed: {e}")
    dxy = None

groups = []
for pair_name, group in raw.groupby("pair", sort=False):
    g = compute_indicators(group)
    if dxy is not None:
        merged = g.join(dxy, how="left")
        g["Corr_DXY_20"] = merged["Log_Returns"].rolling(20).corr(merged["DXY"].pct_change())
    groups.append(g)

df = pd.concat(groups).sort_index()
feature_cols = get_feature_cols(df)
print(f"Feature shape: {df.shape}, features: {len(feature_cols)}")

df[feature_cols] = df[feature_cols].shift(1)
df = df.dropna()
print(f"After shift + dropna: {len(df)} rows")

## Generate ensemble predictions

In [ ]:
# Load feature names from model training
import json
try:
    with open("/kaggle/input/forex-ml-03-model-training/feature_names.json") as f:
        model_features = json.load(f)
    print(f"Loaded {len(model_features)} model feature names")
    # Use model's internal feature names for exact alignment
    if hasattr(model, "feature_names_in_"):
        feature_cols = list(model.feature_names_in_)
        print(f"Using {len(feature_cols)} features from model")
    else:
        # Align: add missing cols as NaN, drop extras, reorder to match model
        for col in model_features:
            if col not in df.columns:
                df[col] = np.nan
        feature_cols = [c for c in model_features if c in df.columns]
        feature_cols = feature_cols[:len(model_features)]
        print(f"Using {len(feature_cols)} features from JSON")
except FileNotFoundError:
    feature_cols = get_feature_cols(df)
    print(f"No feature_names.json found, using {len(feature_cols)} features from data")

X = df[feature_cols]
xgb_probs = model.predict_proba(X)[:, 1]

# Ensemble: if weight < 1.0, blend with equal-probability baseline (TFT not available live)
# In production, you'd also run TFT inference here. For now, use XGBoost with weight adjustment:
if ensemble_weight < 1.0:
    baseline_probs = np.full_like(xgb_probs, 0.5)
    ensemble_probs = ensemble_weight * xgb_probs + (1 - ensemble_weight) * baseline_probs
    print(f"Blended with baseline (weight={ensemble_weight:.3f})")
else:
    ensemble_probs = xgb_probs

signals = np.where(ensemble_probs >= threshold, 1, -1)
long_count = (signals == 1).sum()
short_count = (signals == -1).sum()
print(f"Signal distribution: Long={long_count}, Short={short_count}, Flat=0")


        print(f"  {pair:8s}: {"LONG  " if sig == 1 else "SHORT "} (conf={r["probability"]:.3f}, close={r["close"]:.5f})")
        sig = int(r["signal"])
        r = latest.loc[pair]
    if pair in latest.index:
for pair in CURRENCY_PAIRS:
print("\nLatest signals (last row per pair):")

latest = df.groupby("pair").last()
df["probability"] = ensemble_probs
df["signal"] = signals



## Position sizing (Kelly Criterion)

In [ ]:
WINS_LOSERS_BALANCE = 0.55
AVG_WIN = 0.008
AVG_LOSS = 0.005

signals_list = []
for pair in CURRENCY_PAIRS:
    if pair not in latest.index:
        continue
    r = latest.loc[pair]
    sig = int(r["signal"])
    conf = float(r["probability"])
    if sig == 0:
        pos_size = 0.0
    else:
        pos_size = position_size(ACCOUNT_VALUE, abs(conf - 0.5) * 2, WINS_LOSERS_BALANCE, AVG_WIN, AVG_LOSS)
    signals_list.append({
        "timestamp": datetime.now().isoformat(),
        "pair": pair,
        "signal": sig,
        "direction": "LONG" if sig == 1 else ("SHORT" if sig == -1 else "FLAT"),
        "confidence": round(conf, 4),
        "position_size_usd": round(pos_size, 2),
        "close_price": float(r["close"]),
    })

signals_df = pd.DataFrame(signals_list)
print("Signals with position sizing:")
print(signals_df.to_string(index=False))

## Save signals

In [ ]:
output_dir = Path("/kaggle/working")
output_path = output_dir / "signals.csv"
signals_df.to_csv(output_path, index=False)
print(f"Signals saved to {output_path}")

df.to_parquet(output_dir / "daily_features.parquet")
print(f"Daily features saved: {output_dir / 'daily_features.parquet'}")

print("\nDone. signals.csv ready for execution.")